# COMP5329 — Deep Learning

**Tutorial — Sequence Modeling Architectures II: Pretrained Transformers & State Space Models**

**Semester 1, 2026**

### Learning Objectives
By the end of this tutorial you will be able to:
1. Describe the **pretraining paradigm** that underpins BERT, GPT, and ViT, and state each model's pretraining objective in one sentence.
2. Explain how **BERT** (encoder-only, masked LM, bidirectional) differs from **GPT** (decoder-only, causal LM, autoregressive) in terms of masking and information flow.
3. **Implement** the ViT patch embedding from scratch — unfold an image into patches, linearly project, prepend a learnable `[CLS]` token, and add a learnable positional encoding.
4. Derive the closed-form unroll of a linear SSM recurrence and recognise it as a **1-D causal convolution**, with kernel $K_i = C\bar A^i \bar B$.
5. **Implement** both the recurrent-form and convolutional-form of the same SSM and verify they produce identical outputs on a random input.
6. Explain what Mamba's **selective SSM** buys (input-dependent $\Delta, B, C$) and what it pays for that power (the convolution form disappears, parallel scan is needed instead).
7. Answer exam-style short-answer questions on ViT patch-size trade-offs, SSM recurrent↔convolutional equivalence, and BERT vs. GPT masking semantics.


### Topic Coverage

Week 8 covers **Sequence Modeling Architectures II — pretrained Transformers and State Space Models**. The full topic list (see `Week8_Self_Study.ipynb`) is:

- ✅ **Pretraining paradigm** — self-supervised learning, transfer to downstream tasks *(tutorial, Part A)*
- ✅ **BERT** — encoder-only, masked language modelling, bidirectional context *(tutorial, Part A narrative; no code)*
- ✅ **GPT** — decoder-only, causal language modelling, autoregressive generation *(tutorial, Part A narrative; no code)*
- ✅ **Vision Transformer (ViT)** — patchification, `[CLS]` token, learnable positional encoding, end-to-end patch embedding implementation *(tutorial, Part B Task 1)*
- ✅ **State Space Models (S4)** — continuous-to-discrete derivation, recurrent ↔ convolutional duality, from-scratch kernel implementation *(tutorial, Part B Task 2)*
- ✅ **Mamba (S6)** — input-dependent selective SSM, why the convolution form breaks, parallel scan *(tutorial, Part A one paragraph; Exam Q3 references this)*
- 📖 **Full BERT / GPT fine-tuning recipes** — `[MASK]` token sampling, next-sentence prediction, KV-caching for generation *(self-study)*
- 📖 **Full ViT training on ImageNet** — AdamW schedule, data augmentation, evaluation *(self-study)*
- 📖 **HiPPO initialisation for SSMs** — why $\bar A$ is not random, orthogonal-polynomial motivation *(self-study)*

The tutorial focuses on **two concrete implementations** — the ViT patch embedding and the SSM recurrent↔convolutional duality — because these are the two ideas that most tightly connect the pretraining story of Part A to the architectures on the exam. BERT, GPT, and Mamba are covered at the conceptual level only; the self-study notebook walks through them in full.

The live session is organised into three parts: **Part A** — tutor walkthrough (narrative, no new code), **Part B** — in-class coding exercise (two models, fill-in-TODO), **Part C** — exam-style Q&A.

---

# Part A · Tutor Review

> **Goal.** By the end of Part A you should be able to (a) state the pretraining objective of BERT, GPT, and ViT in one sentence each, (b) sketch the ViT patch-embedding pipeline on a whiteboard, and (c) derive the SSM convolution kernel $K_i = C\bar A^i \bar B$ from the discrete recurrence.

---
## §0 — The Pretraining Paradigm

Before 2018, most deep-learning models were trained **from scratch on the target task**: a sentiment classifier saw only sentiment labels, an image classifier saw only ImageNet labels. This is data-hungry (millions of labelled examples per task) and task-brittle (nothing transfers).

The **pretraining paradigm** flips this:

1. **Pretrain** a large Transformer on a *self-supervised* objective over a huge unlabelled corpus (Wikipedia, Common Crawl, JFT-300M). No human labels needed.
2. **Fine-tune** (or prompt) the pretrained model on the small downstream task — classification, QA, generation, segmentation.

The magic is that the pretrained weights encode enough general structure (grammar, world knowledge, visual textures) that the downstream task only needs a small labelled set to adapt. Three models define the era:

| Model | Architecture | Pretraining Objective | Signature property |
|---|---|---|---|
| **BERT** (2018) | Transformer **encoder** only | Masked language modelling (MLM) | **Bidirectional** context |
| **GPT** (2018+) | Transformer **decoder** only | Causal language modelling (next-token) | **Autoregressive** generation |
| **ViT** (2020) | Transformer **encoder** only | Supervised classification on JFT-300M | **Patches as tokens** |

---
## §1 — BERT: Bidirectional Encoder, Masked LM

BERT = **B**idirectional **E**ncoder **R**epresentations from **T**ransformers.

**The encoder-only choice.** BERT stacks only the encoder half of the Transformer. An encoder block uses **unmasked** self-attention: token $i$ attends to every token in the sequence, both to its left and to its right. This gives each token a representation that is informed by the full bidirectional context — great for understanding tasks (classification, QA, NER) but no good for free-form generation, because during generation you *don't yet have* the right-hand context.

**Masked language modelling (MLM).** If we just trained a bidirectional encoder to reconstruct the input, the trivial solution is the identity function. BERT fixes this by **masking 15% of the tokens** with a special `[MASK]` symbol and asking the model to predict the original tokens from the unmasked context. Because attention is bidirectional, the model learns to use both left and right context to fill in the blank.

**Example.**
```
Input  :  The [MASK] sat on the mat.
Target :            cat
```

The loss is cross-entropy on the masked positions only (about 15% of tokens), using the encoder's hidden state at those positions to predict over the full vocabulary.

**Fine-tuning.** For a downstream task (e.g. sentiment classification), you prepend a `[CLS]` token, run BERT, and attach a small classifier head on top of the `[CLS]` hidden state. All weights — BERT and the head — are fine-tuned together on the labelled task data. No architecture changes, just a new final layer.

---
## §2 — GPT: Causal Decoder, Autoregressive LM

GPT = **G**enerative **P**retrained **T**ransformer.

**The decoder-only choice.** GPT stacks only the decoder half of the Transformer — but with the cross-attention layer removed, since there is no encoder to attend to. The signature of the decoder is **causal (masked) self-attention**: token $i$ can only attend to tokens $\le i$. Concretely, the attention logits are masked with a lower-triangular matrix before the softmax, so the upper triangle becomes $-\infty$ and contributes zero weight.

**Causal language modelling (next-token prediction).** GPT is trained to predict the next token given the prefix:
$$\mathcal L = -\sum_{t=1}^{T}\log p_\theta(x_t \mid x_{<t})$$
This is exactly the classical language-modelling objective, which is why GPT naturally produces fluent text: sampling $x_{t+1} \sim p_\theta(\cdot\mid x_{\le t})$ is a direct use of the training objective at inference.

**Key contrast with BERT.** The masking pattern **is the architecture**:

| Aspect | BERT | GPT |
|---|---|---|
| Attention mask | None (bidirectional) | Lower-triangular (causal) |
| Pretraining objective | Predict `[MASK]` tokens | Predict next token |
| Context at position $t$ | Full sequence | Prefix $x_{<t}$ only |
| Natural use case | Understanding (classify, QA) | Generation (chat, completion) |
| Inference is generative? | No (one forward pass) | Yes (sample one token at a time) |

Exam Q3 will ask you to articulate this contrast — if you can write the masking matrices, you understand the models.

---
## §3 — ViT: From Pixels to Tokens

The Transformer was designed for 1-D sequences of tokens, but images are 2-D pixel grids. **Vision Transformer (ViT)** answers the obvious question: *can we just serialise an image into a sequence and run a standard Transformer encoder on it?* The answer is "yes, if the sequence is short enough and the dataset is large enough."

**Patchification.** An image $X \in \mathbb R^{C \times H \times W}$ is split into non-overlapping patches of size $P \times P$:

$$T = \frac{H}{P} \cdot \frac{W}{P} \quad \text{patches}, \qquad \text{each of shape } C \times P \times P$$

Each patch is **flattened** into a vector of length $C P^2$ and **linearly projected** to an embedding dimension $D$. The output is a sequence of $T$ token vectors, shape $(T, D)$ — exactly the shape a Transformer expects.

**Why not just use pixels as tokens?** With $H=W=224$, a pixel sequence has length $T = 224^2 = 50{,}176$. Self-attention is $O(T^2)$, so pixel-level attention costs $\sim 2.5 \cdot 10^9$ operations *per layer*. With patches of size $P=16$, $T = 14^2 = 196$, and attention costs drop by a factor of $(50176/196)^2 \approx 65{,}000$. Patches are a pragmatic compromise between spatial resolution and compute.

**`[CLS]` token.** BERT-style `[CLS]` is prepended: a single learnable vector $z_{\text{cls}} \in \mathbb R^D$ that has no corresponding patch. After the encoder, the `[CLS]` position's hidden state is used as the image's global representation and fed to a classification head. The token starts off information-free but *pulls in* information from every patch via self-attention at every layer — by the final layer, it carries a learned summary of the whole image.

**Learnable positional encoding.** Unlike CNNs, self-attention is **permutation-invariant** — shuffle the patches and you get the same set of outputs in a different order. To restore spatial awareness, ViT adds a **learnable positional embedding** $E_{\text{pos}} \in \mathbb R^{(T+1) \times D}$ to the token sequence (including the `[CLS]` slot). No sinusoidal formula, no 2-D prior — pure learned lookup, one vector per patch index. This is enough for the model to learn spatial relations from data.

**LayerNorm-first (pre-norm).** ViT uses the **pre-norm** variant of the Transformer block, where LayerNorm is applied *before* each sub-layer rather than after it:
$$z = x + \text{Attn}(\text{LN}(x)), \qquad z' = z + \text{MLP}(\text{LN}(z))$$
Pre-norm is much more stable at depth: the residual path remains un-normalised and gradients flow cleanly from the loss back to the patch embedding. Post-norm Transformers (the original 2017 paper) required careful learning-rate warmup to avoid divergence; pre-norm lets you drop the warmup and train deeper models.

**Why does ViT underperform CNNs on small datasets?** Because CNNs bake in **locality**, **translation equivariance**, and **hierarchical composition** as hard-coded inductive biases, while ViT must learn all three from data. With ImageNet-1K (~1.2M images), ViT cannot learn enough and underperforms. With JFT-300M (~300M images), ViT *surpasses* CNNs because its lack of hard priors becomes an advantage — it can learn long-range, global spatial structure that CNN priors rule out. Exam Q1 will push on this trade-off.

**Part B Task 1** will have you build the ViT patch-embedding pipeline end-to-end.

---
## §4 — State Space Models: From Continuous Dynamics to a Convolution

Transformers are great but expensive: self-attention is $O(T^2)$ in sequence length. For long sequences (audio, DNA, genomics, long documents), we want something cheaper. **State Space Models** (S4, Mamba) achieve $O(T)$ — sometimes $O(T\log T)$ — by borrowing an idea from classical control theory.

### 4.1 Continuous-time linear system

Starting point: a **continuous-time linear state space model** describes how an internal state $h(t) \in \mathbb R^N$ evolves under a scalar input $u(t)$ and produces a scalar output $y(t)$:

$$h'(t) = A\,h(t) + B\,u(t), \qquad y(t) = C\,h(t)$$

Here $A \in \mathbb R^{N \times N}$, $B \in \mathbb R^{N \times 1}$, $C \in \mathbb R^{1 \times N}$. This is the basic Kalman-filter equation, and it has been used in signal processing, control, and econometrics since the 1960s.

### 4.2 Discretization (ZOH / Euler)

To run this on discrete data (tokens, samples), we discretise at step size $\Delta$. The simplest scheme is **zero-order hold (ZOH)** — or, even simpler, a **first-order Euler step**:

$$h_{k+1} \approx h_k + \Delta \big(A\,h_k + B\,u_k\big) = (I + \Delta A)\,h_k + \Delta B\,u_k$$

Absorbing constants, define
$$\bar A = I + \Delta A, \qquad \bar B = \Delta B$$
and we arrive at the **discrete SSM**:

$$\boxed{\,h_{k+1} = \bar A\,h_k + \bar B\,u_k, \qquad y_k = C\,h_k\,}$$

This is **a linear RNN**. Same shape as a Vanilla RNN from Week 7, *minus* the $\tanh$ non-linearity. That missing non-linearity is load-bearing — everything that follows depends on it.

### 4.3 Recurrent form → convolutional form

Unroll the recurrence with $h_0 = 0$:
$$h_1 = \bar B\,u_0, \quad h_2 = \bar A \bar B\,u_0 + \bar B\,u_1, \quad h_3 = \bar A^2 \bar B\,u_0 + \bar A \bar B\,u_1 + \bar B\,u_2, \ldots$$
In closed form:
$$h_k = \sum_{i=0}^{k-1} \bar A^{\,k-1-i}\,\bar B\,u_i \quad\Longrightarrow\quad y_k = C h_k = \sum_{i=0}^{k-1} \underbrace{\big(C\,\bar A^{\,k-1-i}\,\bar B\big)}_{K_{k-1-i}}\,u_i$$

Define the length-$L$ **SSM convolution kernel**
$$\boxed{\,K_i = C\,\bar A^{\,i}\,\bar B, \qquad i = 0, 1, \ldots, L-1\,}$$
Then $y = u * K$ — **a single 1-D causal convolution**. The whole SSM collapses into one convolution on the GPU.

### 4.4 Why this matters

The duality gives you the **best of both worlds**:

| Form | Compute | Memory at inference | Parallelism |
|---|---|---|---|
| **Recurrent** $h_{k+1} = \bar A h_k + \bar B u_k$ | $O(L)$ steps | $O(N)$ state | Sequential (RNN-style) |
| **Convolutional** $y = u * K$ | $O(L \log L)$ with FFT | $O(L)$ kernel | Fully parallel (GPU-friendly) |

During **training** you use the convolution form for fully-parallel FLOPs on a GPU. During **inference on a streaming input** you use the recurrent form because it only carries an $N$-dimensional state forward (no need to materialise the full kernel or cache the whole history). Same weights, two ways of computing.

**Part B Task 2** will have you build $K_i$ and verify $y = u * K$ numerically against the recurrence.

---
## §5 — Mamba: Selective SSM *(conceptual, one paragraph)*

S4 is great but has a limitation: because $\bar A, \bar B, C$ are constants (independent of the input), the SSM applies the *same* linear dynamics to every token. It cannot "pay attention" to a specific input the way a Transformer can. **Mamba (S6)** fixes this by making $\Delta, B, C$ into **functions of the current input** $u_k$ — so each time-step has its own discretisation step and its own projection matrices. This is *selectivity*: the model can choose to update or ignore its state based on what it sees. But this has a cost: since $\bar A^{\,k-i}$ now depends on all intermediate inputs, **the convolution kernel $K_i$ is no longer well-defined** — $y_t$ no longer decomposes as a fixed-kernel convolution. Mamba recovers parallel training via a hardware-aware **parallel scan** (associative scan on the per-step linear operators), which is $O(L\log L)$ tree-depth but materialises no kernel. Exam Q3 will push on this — if you understand that selectivity destroys the convolution form, you understand Mamba.

---

# Part B · In-Class Exercise

> **Your job**: fill in the `# TODO` blocks in the two tasks below. Each task has a collapsed **Solution** cell underneath — try the task yourself first, then expand the solution to compare.
>
> **Exam-ready standard**: after solving each task, you should be able to reproduce the code from a blank cell using only the markdown description. These are candidate exam questions.

## Task B1 · Vision Transformer — Patch Embedding

The first piece of a ViT is the **patch embedding**: turn an image $(C, H, W)$ into a sequence $(T+1, D)$ of token vectors, where the $+1$ is the prepended `[CLS]` token.

**Pipeline:**

1. **Unfold** the image into non-overlapping $P \times P$ patches. `nn.Unfold` returns shape $(B, C\cdot P^2, T)$ where $T = (H/P)(W/P)$.
2. **Linearly project** each patch from $C\cdot P^2$ to embedding dimension $D$. This is a single `nn.Linear`.
3. **Prepend** a learnable `[CLS]` token — one vector of shape $(1, 1, D)$ broadcast across the batch.
4. **Add** a learnable positional embedding of shape $(1, T+1, D)$, also broadcast across the batch.

Output: a tensor of shape $(B, T+1, D)$, ready for a standard Transformer encoder.

**Setup (read-only — do not modify):**

In [ ]:
# Setup (read-only — do not modify)
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

# Toy image batch: 2 RGB images, 16x16, split into 4x4 patches.
B, C, H, W = 2, 3, 16, 16
P          = 4                       # patch size
D          = 32                      # embedding dimension
T          = (H // P) * (W // P)     # number of patches = 16

print(f"Image batch: (B={B}, C={C}, H={H}, W={W})")
print(f"Patch size P = {P}  →  T = {T} patches per image")
print(f"Embedding dimension D = {D}")
print(f"Expected output shape: (B, T+1, D) = ({B}, {T+1}, {D})")

### Task B1 — Fill in `ViTPatchEmbedding`

Complete the four `# TODO` blocks in the module below. Do not change the `__init__`; your job is the `forward`.

In [ ]:
class ViTPatchEmbedding(nn.Module):
    """ViT patch embedding: image → (B, T+1, D) token sequence.

    Forward pipeline
    ----------------
    image (B, C, H, W)
       → unfold to (B, C*P*P, T)
       → transpose / project to (B, T, D)
       → prepend [CLS] token to get (B, T+1, D)
       → add positional embedding → (B, T+1, D)
    """

    def __init__(self, img_size: int, patch_size: int, in_chans: int, embed_dim: int):
        super().__init__()
        assert img_size % patch_size == 0, "img_size must be divisible by patch_size"
        self.img_size   = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        self.embed_dim  = embed_dim

        # Unfold breaks the image into non-overlapping P×P patches and flattens them.
        self.unfold = nn.Unfold(kernel_size=patch_size, stride=patch_size)

        # Linear projection: each flattened patch (C*P*P) → D
        self.proj = nn.Linear(in_chans * patch_size * patch_size, embed_dim)

        # Learnable [CLS] token — one vector, broadcast across the batch
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))

        # Learnable positional embedding — one vector per (CLS + patch) slot
        self.pos_embed = nn.Parameter(torch.zeros(1, 1 + self.num_patches, embed_dim))

        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B = x.shape[0]

        # TODO 1 — unfold the image into patches.
        #          Input  shape: (B, C, H, W)
        #          Output shape: (B, C*P*P, T)   where T = (H/P)*(W/P)
        patches = ...

        # TODO 2 — rearrange and linearly project each patch to D.
        #          After transpose: (B, T, C*P*P)
        #          After self.proj: (B, T, D)
        tokens = ...

        # TODO 3 — prepend the [CLS] token.
        #          Expand self.cls_token from (1, 1, D) to (B, 1, D), then concatenate.
        #          Output shape: (B, T+1, D)
        cls = ...
        tokens = ...

        # TODO 4 — add the learnable positional embedding.
        #          self.pos_embed has shape (1, T+1, D); PyTorch broadcasts over batch.
        tokens = ...

        return tokens


<details>
<summary><b>▸ Solution · Task B1</b> (click to expand)</summary>

```python
def forward(self, x: torch.Tensor) -> torch.Tensor:
    B = x.shape[0]

    # TODO 1 — unfold:  (B, C, H, W) → (B, C*P*P, T)
    patches = self.unfold(x)

    # TODO 2 — transpose to put T in the middle, then project to D
    #          (B, C*P*P, T) → (B, T, C*P*P) → (B, T, D)
    tokens = self.proj(patches.transpose(1, 2))

    # TODO 3 — prepend the [CLS] token.  (1,1,D) → (B,1,D), then concat along dim=1
    cls = self.cls_token.expand(B, -1, -1)
    tokens = torch.cat([cls, tokens], dim=1)     # (B, T+1, D)

    # TODO 4 — add positional embedding (broadcasts over B)
    tokens = tokens + self.pos_embed

    return tokens
```

**Key points the tutor will highlight:**

- **Why unfold?** `nn.Unfold` with `kernel_size = stride = P` is the tensor-fluent way to grab non-overlapping patches. It is equivalent to a `Conv2d(C, C*P*P, kernel_size=P, stride=P, groups=C)` with identity weights, but cheaper and non-learnable. In practice, many ViT codebases actually use a `Conv2d(C, D, kernel_size=P, stride=P)` and fold the linear projection into the conv itself — that's the same operation, just a different implementation.

- **Why `.transpose(1, 2)` before the Linear?** `nn.Linear` acts on the **last** dimension. After unfold, the patch-vector axis is dim=1 (shape `(B, C*P*P, T)`). Transposing to `(B, T, C*P*P)` puts the patch-vector on the last axis, so `self.proj` applies element-wise across all $T$ tokens with the same shared weight. This is exactly the weight-sharing story from Week 5 CNNs, ported to ViT.

- **Why a learnable `[CLS]` token, not the mean of patches?** The `[CLS]` starts off information-free (initialised near zero) but receives updates from every patch at every layer via self-attention. By the final layer it has learned an *aggregation function* that is better than a fixed mean — the model decides which patches matter and how to combine them. Using a fixed mean-pool instead is actually competitive (see "GAP head" in the ViT paper), but `[CLS]` keeps the architecture aligned with BERT and is slightly more flexible.

- **Why learnable positional encoding, not sinusoidal?** Self-attention is permutation-invariant: shuffle the tokens and the output set is unchanged (up to the shuffle). Without positional information, ViT could not distinguish a cat on the left from a cat on the right. ViT uses a **fully learnable** embedding `(1, T+1, D)` rather than the sinusoidal encoding of the original Transformer because (i) it gives the model freedom to learn a 2-D spatial structure from data, and (ii) ViT always sees a fixed number of patches at training time, so there is no need for the length-agnostic property that sinusoidal encodings buy.

- **Why LayerNorm-first (pre-norm)?** Not implemented in *this* module, but essential for the encoder block that follows. In the pre-norm block `z = x + Attn(LN(x))`, the residual path `x` is never normalised, so gradients flow cleanly from the loss back to the patch embedding with no vanishing. Post-norm (original Transformer, 2017) puts LayerNorm *after* the residual add and requires careful learning-rate warmup. ViT uses pre-norm for stability at depth.

- **Shapes on the demo below**: $(B=2, C=3, H=16, W=16)$ with $P=4$ gives $T=(16/4)^2=16$ patches → `(2, 17, 32)` after `[CLS]`.
</details>

### Demo: verify the output shape

The code cell below instantiates `ViTPatchEmbedding`, feeds it a toy batch, and prints the shapes. After Task B1 is correct, you should see `(2, 17, 32)`.

In [ ]:
# Demo — feed a toy batch through the patch embedding and check shapes
vit_embed = ViTPatchEmbedding(img_size=H, patch_size=P, in_chans=C, embed_dim=D)

dummy_images = torch.randn(B, C, H, W)
tokens = vit_embed(dummy_images)

print(f"Input  shape : {tuple(dummy_images.shape)}")
print(f"Output shape : {tuple(tokens.shape)}   (B, T+1, D) with [CLS] prepended")
print()
print(f"Number of parameters:")
print(f"  proj      : {sum(p.numel() for p in vit_embed.proj.parameters()):>6d}")
print(f"  cls_token : {vit_embed.cls_token.numel():>6d}")
print(f"  pos_embed : {vit_embed.pos_embed.numel():>6d}")

### Demo 2: compose one Transformer encoder block on top

Patch embedding alone is not a ViT — it is just the tokenizer. A full ViT stacks many Transformer encoder blocks on top. To keep this tutorial focused on the patchification idea, we use PyTorch's built-in `nn.TransformerEncoderLayer` (with `norm_first=True` for the pre-norm variant) and run a single block to show the shape is preserved and the `[CLS]` position can be read out as a global image representation.

**No TODO here — this is illustration only.**

In [ ]:
# Demo 2 — one Transformer encoder block on top of the patch embedding
encoder_block = nn.TransformerEncoderLayer(
    d_model=D, nhead=4, dim_feedforward=4*D,
    batch_first=True, norm_first=True,      # norm_first=True = pre-norm variant
)

out = encoder_block(tokens)                 # (B, T+1, D)
cls_repr = out[:, 0, :]                     # (B, D) — the [CLS] position

print(f"After encoder block : {tuple(out.shape)}")
print(f"[CLS] representation : {tuple(cls_repr.shape)}")
print(f"On top of cls_repr you would attach a small Linear(D, num_classes) classifier head.")

---

## Task B2 · State Space Model — Recurrent ↔ Convolutional Duality

You will write **two functions** that both compute $y_{0:L-1}$ from the same input $u_{0:L-1}$ and the same matrices $\bar A,\ \bar B,\ C$:

- `ssm_recurrent` — steps through the recurrence token-by-token (the "RNN view"). $O(N)$ state per step.
- `ssm_conv` — builds a length-$L$ kernel $K$ with $K_i = C\bar A^i\bar B$ and computes $y = u * K$ as a single 1-D causal convolution (the "parallel training view").

When both are correct, the `assert` at the bottom turns green: the two forms are numerically identical on a random test input.

**Setup (read-only — do not modify):**

In [ ]:
# Setup (read-only — do not modify)
torch.manual_seed(0)
L, N = 20, 4                       # sequence length L, state dimension N

# A small stable SSM — spectral radius of A_bar < 1 so the T=20 unroll does not blow up.
A_bar = 0.9 * torch.eye(N) + 0.01 * torch.randn(N, N)
B_bar = torch.randn(N, 1)          # (N, 1) — one scalar input channel
C_mat = torch.randn(1, N)          # (1, N) — one scalar output channel
u     = torch.randn(L)             # scalar input sequence

spectral_radius = torch.linalg.eigvals(A_bar).abs().max().item()
print(f"A_bar shape: {tuple(A_bar.shape)}, spectral radius ≈ {spectral_radius:.3f}")
print(f"B_bar shape: {tuple(B_bar.shape)}, C shape: {tuple(C_mat.shape)}")
print(f"u shape: {tuple(u.shape)}")

### Task B2a · Recurrent form

Step through the recurrence one token at a time. State $h$ has shape `(N,)`; update it in place:

$$h_{k+1} = \bar A\,h_k + \bar B\,u_k, \qquad y_k = C\,h_k$$

(We use the convention $y_k = C h_k$ **after** the state has absorbed $u_k$, so output $y_0$ already sees $u_0$. This matches the closed-form derivation $y_k = \sum_{i \le k} K_{k-i}\,u_i$.)

In [ ]:
def ssm_recurrent(A_bar, B_bar, C_mat, u):
    """Linear SSM in recurrent form. O(N) state per step — RNN-style."""
    L_ = u.shape[0]
    N_ = A_bar.shape[0]
    h = torch.zeros(N_)
    y = torch.zeros(L_)
    b = B_bar.squeeze(-1)                    # (N,), one scalar input channel
    for k in range(L_):
        # TODO 1 — update state: h ← A_bar @ h + b * u[k]
        h = ...

        # TODO 2 — emit output: y[k] = (C_mat @ h).squeeze()
        y[k] = ...
    return y


<details>
<summary><b>▸ Solution · Task B2a</b> (click to expand)</summary>

```python
def ssm_recurrent(A_bar, B_bar, C_mat, u):
    L_ = u.shape[0]
    N_ = A_bar.shape[0]
    h = torch.zeros(N_)
    y = torch.zeros(L_)
    b = B_bar.squeeze(-1)
    for k in range(L_):
        h    = A_bar @ h + b * u[k]          # TODO 1 — state update
        y[k] = (C_mat @ h).squeeze()         # TODO 2 — output projection
    return y
```

**Key points:**

- **`b = B_bar.squeeze(-1)`** converts `(N, 1)` to `(N,)` so that `b * u[k]` is a scalar-scaled vector (shape `(N,)`). Without the squeeze, you would need `(B_bar * u[k]).squeeze(-1)` at every step — functionally identical, just uglier.
- **`C_mat @ h`** produces a shape-`(1,)` tensor (because `C_mat` is `(1, N)`); `.squeeze()` makes it a scalar so the assignment to `y[k]` works.
- **Memory**: at every step only $h$ of shape `(N,)` lives in memory. This is the RNN-style $O(N)$-per-step advantage that SSMs inherit from their recurrent form — at inference time on a streaming input, you do *not* need to remember the past $u_0, \ldots, u_{k-1}$.
- **Why no `tanh`?** That's the load-bearing difference between an SSM and a Vanilla RNN, and it is exactly what lets us collapse the recurrence into a convolution below. Put a $\tanh$ here and Task B2b becomes impossible.
</details>

### Task B2b · Convolutional form — build the kernel

**Derivation you should do on paper first** (and it is on the lecture slides).

Unroll the recurrence with $h_0 = 0$:
$$h_1 = \bar B\,u_0, \quad h_2 = \bar A\bar B\,u_0 + \bar B\,u_1, \quad h_3 = \bar A^2\bar B\,u_0 + \bar A\bar B\,u_1 + \bar B\,u_2, \ldots$$

In closed form:
$$h_{k+1} = \sum_{i=0}^{k} \bar A^{\,k-i}\,\bar B\,u_i \quad\Longrightarrow\quad y_k = C h_{k+1} = \sum_{i=0}^{k}\underbrace{\big(C\,\bar A^{\,k-i}\,\bar B\big)}_{K_{k-i}}\,u_i$$

So if you define the length-$L$ kernel $K$ with entries $K_i = C\,\bar A^{\,i}\,\bar B$, then $y_k = (K * u)_k$ — a single 1-D **causal** convolution.

**Two efficiency notes:**

1. Compute $K$ **incrementally**. Maintain a running tensor `A_pow_B` of shape `(N, 1)`, starting at $\bar B$, and left-multiply by $\bar A$ at each step to get $\bar A\bar B, \bar A^2\bar B, \ldots$. Do *not* call `torch.matrix_power(A_bar, i)` inside a loop — that is quadratic work on a linear problem.
2. For the convolution itself, a double loop is fine for $L=20$. A faster alternative is `F.conv1d` with the kernel **flipped** and the input left-padded by $L-1$.

In [ ]:
def build_ssm_kernel(A_bar, B_bar, C_mat, L_):
    """Build the length-L convolution kernel  K[i] = C @ A_bar^i @ B_bar."""
    N_ = A_bar.shape[0]
    K = torch.zeros(L_)

    # TODO 1 — initialise the running product  A_pow_B = B_bar  (shape (N, 1))
    A_pow_B = ...

    # TODO 2 — loop i = 0 .. L-1:
    #            K[i] = (C_mat @ A_pow_B).squeeze()   # scalar
    #            A_pow_B = A_bar @ A_pow_B            # advance one power of A_bar
    for i in range(L_):
        K[i] = ...
        A_pow_B = ...

    return K


def ssm_conv(u, kernel):
    """Apply the SSM kernel as a 1-D causal convolution:  y[k] = sum_{i<=k} K[k-i] * u[i]."""
    L_ = u.shape[0]
    y = torch.zeros(L_)

    # TODO 3 — causal convolution (double loop is fine for L=20)
    #           y[k] = sum_{i=0..k} kernel[k-i] * u[i]
    for k in range(L_):
        y[k] = ...

    return y


<details>
<summary><b>▸ Solution · Task B2b</b> (click to expand)</summary>

```python
def build_ssm_kernel(A_bar, B_bar, C_mat, L_):
    N_ = A_bar.shape[0]
    K = torch.zeros(L_)
    A_pow_B = B_bar                              # TODO 1 — start at B_bar, shape (N, 1)
    for i in range(L_):
        K[i]    = (C_mat @ A_pow_B).squeeze()    # TODO 2a — scalar K[i] = C A^i B
        A_pow_B = A_bar @ A_pow_B                # TODO 2b — advance one power of A_bar
    return K


def ssm_conv(u, kernel):
    L_ = u.shape[0]
    y = torch.zeros(L_)
    for k in range(L_):
        y[k] = sum(kernel[k - i] * u[i] for i in range(k + 1))   # TODO 3 — causal
    return y
```

**Key points:**

- **Incremental kernel build is $O(L\cdot N^2)$**, not $O(L^2\cdot N^3)$. The trick is that $\bar A^{i+1}\bar B = \bar A\,(\bar A^i\bar B)$, so each new kernel entry costs one matrix-vector multiply `(N,N) · (N,1) = (N,1)`. Doing `torch.matrix_power(A_bar, i) @ B_bar` inside the loop is the naive $O(L^2 N^3)$ version and is exactly the anti-pattern the efficiency note warned against.

- **Causality**. The sum $y_k = \sum_{i=0}^{k} K_{k-i}\,u_i$ only touches $u_i$ for $i \le k$, so the convolution is **causal** — each output only depends on past inputs. This matches the recurrence, which only ever sees the current and past inputs.

- **Shape `K[i] = (C_mat @ A_pow_B).squeeze()`**. $C_{\text{mat}}$ is `(1, N)` and `A_pow_B` is `(N, 1)`, so the product is `(1, 1)`; `.squeeze()` makes it a scalar that slots into the scalar tensor `K[i]`.

- **Why this enables parallel training.** At training time we know $u_{0:L-1}$ all at once. Computing $y$ token-by-token via the recurrence forces a *sequential* chain of $L$ matrix-vector multiplies — no parallelism along time, bad for GPUs. Computing $y$ as a single 1-D convolution is embarrassingly parallel: every output position can be computed independently once the kernel is known. On GPUs you can then use `F.conv1d` (or even FFT-based convolution for $O(L\log L)$).

- **What Mamba gives up.** If $\bar A, \bar B, C$ become functions of $u_k$ (per-step matrices), then $\bar A^{\,k-i}$ is no longer well-defined — each step has its own $\bar A$, so you cannot raise it to a power. The relationship $y_k = \sum_i K_{k-i}\,u_i$ with a fixed $K$ dissolves. Mamba has to give up the convolution form entirely and recovers parallel training via a **parallel scan** instead.
</details>

### Verification

Run both forms and check they agree numerically. After Task B2a and Task B2b are both correct, the `assert` will pass and you will see `✓ recurrent and convolutional forms agree`.

In [ ]:
# Auto-verify — turns green once both Task B2a and Task B2b are correct.
y_rec  = ssm_recurrent(A_bar, B_bar, C_mat, u)
kernel = build_ssm_kernel(A_bar, B_bar, C_mat, L)
y_conv = ssm_conv(u, kernel)

print(f"kernel[:5] = {kernel[:5].tolist()}")
print(f"y_rec[:5]  = {y_rec[:5].tolist()}")
print(f"y_conv[:5] = {y_conv[:5].tolist()}")

assert torch.allclose(y_rec, y_conv, atol=1e-4), "recurrent form != convolutional form"
print()
print("✓ recurrent and convolutional forms agree on L=20")


### Bonus · Three reflection prompts *(for fast finishers)*

Don't start these until the `assert` above passes. They are deliberately open-ended — write your answers in the comments of the cell below (or discuss with a neighbour).

In [ ]:
# Bonus 1 — stability
# Try replacing A_bar with `1.05 * torch.eye(N)` (spectral radius > 1).
# Re-run ssm_recurrent and ssm_conv on the same u.
# Which form overflows first, and why? How does this motivate HiPPO initialisation?
# (The self-study notebook has the full HiPPO answer.)

# Bonus 2 — long sequences
# Set L = 10_000 and re-run both forms. Compare the peak memory of each:
#   - recurrent: keeps only h of shape (N,)        → O(N) memory
#   - convolutional: materialises kernel of length L  → O(L) memory
# For L = 10^6 on a streaming inference task, which form do you want?

# Bonus 3 — what Mamba gives up
# Suppose A_bar, B_bar, C_mat each become a function of u[k] (per-step matrices).
# Can you still write K[i] = C @ A_bar^i @ B_bar and get a single convolution? Why not?
# Trace it on L=3 with distinct A^{(0)}, A^{(1)}, A^{(2)} and see where the exponent
# A_bar^i stops making sense. This is precisely why Mamba abandons the convolution form
# and uses a parallel scan instead.


---
# Part C · Exam-Style Questions

> Three short-answer questions at medium-high difficulty. Each has a collapsed **Model Answer** cell directly below — attempt each question on paper first, then expand the solution.

---

## Q1 · ViT patch-size trade-off and inductive bias *(Part B Task 1)*

A Vision Transformer splits a $224 \times 224$ image into non-overlapping $P \times P$ patches, linearly projects each patch to dimension $D$, prepends a learnable `[CLS]` token, adds a learnable positional encoding, and feeds the resulting $(T+1, D)$ sequence into a standard Transformer encoder.

**(a)** Suppose we change the patch size from $P=16$ to $P=32$. Compute explicitly how (i) the sequence length $T$, (ii) the FLOPs of a single self-attention layer (which scale as $T^2 D$), and (iii) the spatial area each token covers change. Use concrete numbers.

**(b)** Explain, in terms of the `forward` pipeline you implemented in Task B1, **exactly why** the learnable positional embedding is necessary. What would go wrong if we simply removed it?

**(c)** ViT underperforms a strong CNN (e.g. EfficientNet-B7) when both are trained only on ImageNet-1K, but *surpasses* it when pre-trained on JFT-300M. Using Week 4's concept of **inductive bias**, explain what built-in assumptions CNNs make that ViT does not, and why those assumptions help on small data but become a ceiling on large data.

*Your answer:*

<details>
<summary><b>▸ Model Answer · Q1</b></summary>

**(a)** With $P=16$: $T = (224/16)^2 = 14^2 = 196$. With $P=32$: $T = (224/32)^2 = 7^2 = 49$. So:

- **(i)** $T$ drops from 196 to 49 — a factor of **4× fewer tokens**.
- **(ii)** Self-attention FLOPs scale as $T^2 D$, so they drop by $(196/49)^2 = 16$× — **16× cheaper attention**.
- **(iii)** The spatial area a single token covers goes from $16 \times 16 = 256$ pixels to $32 \times 32 = 1024$ pixels — **4× coarser** spatial granularity.

Trade-off: larger patches = cheaper attention + coarser spatial detail; smaller patches = more expressive + quadratically more expensive. This is exactly the compute–accuracy dial ViT exposes.

**(b)** In the Task B1 pipeline, the patch embedding after step 3 is `(B, T+1, D)`, a **set of tokens** with no intrinsic spatial ordering — unfold produces them in row-major order, but a Transformer encoder sees them as an unordered multiset because self-attention is **permutation-equivariant**: shuffle the token sequence and you get the same output set in the shuffled order. Without a positional embedding, the model cannot tell "the patch at the top-left of the image" apart from "the patch at the bottom-right." A ViT without positional encoding would be invariant to permutations of the 16 patches — a cat centred in the image and a cat whose patches were randomly shuffled would produce *identical* hidden states. The learnable `pos_embed` breaks this symmetry by adding a distinct vector to each slot, turning a permutation-invariant set into a structured sequence.

**(c)** Convolutional layers bake in three inductive biases:
1. **Locality** — a filter only looks at a small neighbourhood.
2. **Translation equivariance** — the same filter slides across the whole image, so a cat in the top-left and a cat in the bottom-right activate the same features (just shifted).
3. **Hierarchical composition** — shallow layers learn textures, deep layers learn semantics.

These are hand-engineered priors matched to natural-image statistics. ViT's self-attention has none of these — it is a global, permutation-respecting pairwise operator, and any spatial structure must be *learned* from data. On **small** datasets (ImageNet-1K ≈ 1.2M images), the CNN's priors are exactly right; ViT cannot see enough images to rediscover them, and underperforms. On **large** datasets (JFT-300M ≈ 300M images), ViT has enough data to learn spatial organisations that are more flexible than the hard-coded CNN priors — including global, long-range dependencies in a single layer — and the rigid CNN priors become a *ceiling*.

**One-line summary**: inductive bias is a double-edged sword — it does your homework for you when data is scarce, and holds you back when data is abundant.
</details>

---

## Q2 · SSM recurrent ↔ convolutional equivalence *(Part B Task 2)*

The discrete SSM is
$$h_{k+1} = \bar A\,h_k + \bar B\,u_k, \qquad y_k = C\,h_k$$
with $\bar A \in \mathbb R^{N\times N}$, $\bar B \in \mathbb R^{N \times 1}$, $C \in \mathbb R^{1 \times N}$.

**(a)** Starting from this recurrence with $h_0 = 0$, derive the closed-form expression for $y_k$ as a function of the inputs $u_{0}, u_1, \ldots, u_k$. Give an explicit formula for the length-$L$ kernel $K$ such that $y = K * u$ (causal convolution).

**(b)** Standard scaled dot-product (softmax) attention computes $y_t = \sum_{i \le t} \alpha_{t,i}\,v_i$ with $\alpha_{t,i} = \exp(q_t^\top k_i)/Z_t$. Explain why this **cannot** be written as $y = K * u$ for any input-independent $K$. Point to the specific algebraic obstruction — two sentences max.

**(c)** In Task B2 you observed that `build_ssm_kernel` uses an *incremental* running product `A_pow_B = A_bar @ A_pow_B` rather than calling `torch.matrix_power(A_bar, i) @ B_bar` at each iteration. Give the asymptotic cost of each approach and explain in one sentence why the incremental version is asymptotically optimal.

*Your answer:*

<details>
<summary><b>▸ Model Answer · Q2</b></summary>

**(a)** Unroll the recurrence with $h_0 = 0$:
$$h_1 = \bar B\,u_0,\quad h_2 = \bar A\bar B\,u_0 + \bar B\,u_1,\quad h_3 = \bar A^2\bar B\,u_0 + \bar A\bar B\,u_1 + \bar B\,u_2,\ \ldots$$
In closed form:
$$h_{k+1} = \sum_{i=0}^{k} \bar A^{\,k-i}\,\bar B\,u_i \quad\Longrightarrow\quad y_k = C h_{k+1} = \sum_{i=0}^{k}\big(C\,\bar A^{\,k-i}\,\bar B\big)\,u_i$$
Define $K_i = C\,\bar A^{\,i}\,\bar B$ for $i = 0, 1, \ldots, L-1$. Then
$$y_k = \sum_{i=0}^{k} K_{k-i}\,u_i = (K * u)_k$$
— a length-$L$ **causal convolution**. The algebraic property that makes this work is that $\bar A, \bar B, C$ are **constants** (independent of $k$ and of $u$), so the weight linking $u_i$ to $y_k$ depends only on the *gap* $k - i$ — precisely the definition of a convolution kernel.

**(b)** The attention weight $\alpha_{t,i}$ depends on $t$ (through $q_t$) and $i$ (through $k_i$) via the inner product $q_t^\top k_i$ — the two indices are **entangled, not separable**, so the weight is not a function of $t - i$ alone. On top of that, $Z_t = \sum_{j \le t}\exp(q_t^\top k_j)$ is a softmax normaliser that depends on the entire prefix and is non-linear — even the numerator $\exp(q_t^\top k_i)\,v_i$ would not separate. No fixed, input-independent $K$ can reproduce these weights, so no convolution form exists.

**(c)**
- **Naive** `torch.matrix_power(A_bar, i) @ B_bar` at each step: computing $\bar A^i$ fresh every iteration costs $O(i \cdot N^3)$, so the whole loop is $O(L^2 N^3)$.
- **Incremental** `A_pow_B = A_bar @ A_pow_B`: each iteration is one matrix-vector multiply `(N,N) · (N,1)`, costing $O(N^2)$. Total is $O(L \cdot N^2)$.

The incremental version is asymptotically optimal because it reuses the previous power: $\bar A^{i+1}\bar B = \bar A\,(\bar A^i \bar B)$, so each new kernel entry costs only one matrix-vector multiply on top of what is already in memory — no wasted work. This is the same "prefix-sum / running product" idea that makes cumulative-sum, dynamic programming, and RNN unrolling efficient.

**Key points the tutor will land:**
- The kernel $K_i = C\bar A^i\bar B$ is literally the **impulse response** of the linear system — drive the SSM with $u_0 = 1, u_{>0} = 0$ and you read off $K$ at the output.
- Convolution-form equivalence is exactly why S4-style SSMs can be trained on TPU/GPU at Transformer speeds, while still running in $O(N)$ per step at inference time.
- What Mamba gives up: making $\bar A, \bar B, C$ depend on $u_k$ destroys both the "constants" hypothesis (so the separable gap-only weight disappears) *and* the incremental-power trick (every step has its own $\bar A$, so $\bar A^i$ is not well-defined). Mamba recovers parallel training via a parallel scan instead.
</details>

---

## Q3 · BERT vs. GPT — masking *is* the architecture *(Part A §1–§2)*

BERT and GPT are both Transformer stacks, both pretrained on massive text corpora, and both produce contextual token representations. Yet one is an *encoder-only* model used for understanding, and the other is a *decoder-only* model used for generation.

**(a)** For a sequence of length $T$, write down the **attention mask** used by BERT and by GPT as a $T \times T$ matrix (or describe it precisely — "entry $(i,j)$ equals 1 if token $i$ can attend to token $j$, else 0"). What shape does each mask have?

**(b)** BERT is pretrained with masked language modelling (MLM): 15% of input tokens are replaced by `[MASK]` and the model predicts the originals. Explain in one sentence why this objective **requires** BERT's bidirectional attention — i.e. why it would fail on a causally-masked GPT.

**(c)** You want to use a pretrained model to do two things:
*(i)* classify movie reviews as positive or negative;
*(ii)* generate a continuation of a user's half-written email.

Which of BERT / GPT is the natural choice for each, and **why**? Your answer must refer to the attention mask from part (a), not just handwave "BERT is for understanding."

*Your answer:*

<details>
<summary><b>▸ Model Answer · Q3</b></summary>

**(a)**

- **BERT** uses a **full (all-ones) mask**: $M_{ij} = 1$ for every $(i, j)$, so every token attends to every other token in both directions. Shape $T \times T$, every entry is 1.
- **GPT** uses a **lower-triangular (causal) mask**: $M_{ij} = 1$ if $j \le i$, else 0. Shape $T \times T$; the upper triangle is zero. In code this is the attention-logit mask
$$M = \begin{pmatrix} 1 & 0 & 0 & \cdots & 0 \\ 1 & 1 & 0 & \cdots & 0 \\ 1 & 1 & 1 & \cdots & 0 \\ \vdots & & & \ddots & \vdots \\ 1 & 1 & 1 & \cdots & 1 \end{pmatrix}$$
applied as additive $-\infty$ on the masked positions before the softmax so they contribute zero weight.

**(b)** MLM replaces a token in the *middle* of the sequence and asks the model to predict it from its context. To get any signal about the original token, the encoder at the masked position must be able to attend to tokens both to its **left** AND **right** — the right-hand context is critical (the cat sat on the \_\_\_ → "mat" only from the left; the \_\_\_ sat on the mat → "cat" only from the right; in practice both are needed). A causally-masked GPT can only see tokens to the *left* of each position, so it literally cannot condition on the right context and the MLM objective is strictly weaker than predicting the next token from the prefix. GPT's causal mask is incompatible with MLM.

**(c)**

- **(i) Movie-review sentiment classification → BERT.** For a classification task you see the *entire* review up front and want the best possible representation at a single `[CLS]` position. BERT's bidirectional attention mask (the all-ones matrix above) lets the `[CLS]` representation at the top of the stack combine information from every word in the review — left context and right context, early and late — in a single forward pass. A GPT `[CLS]` at position 0 could never see a single word of the review (upper triangle zeroed), and a GPT `[CLS]` at position $T$ could see the whole review but with the strict ordering constraint that forces every attention weight to be a function of a one-way prefix, losing the symmetric pair-interactions BERT has. Bidirectional = better for "look at everything at once" tasks.

- **(ii) Email-continuation generation → GPT.** Here you are given a prefix and must produce tokens one at a time. This is literally the objective GPT was pretrained on: $p_\theta(x_t \mid x_{<t})$ with a causal mask. At inference you feed the prefix, sample $x_{T+1}$, append it, and repeat — the causal mask guarantees that the prediction at each new step only looks at tokens that already exist, so the sampling procedure is consistent with training. You could in principle fine-tune BERT for generation, but the MLM objective does not match how you'd actually call the model at inference (there's no `[MASK]` in the user's half-written email, and BERT was never trained to produce coherent long sequences), so you would need architectural hacks that defeat the point of pretraining. Causal mask = naturally fits autoregressive generation.

**Key points:**
- The attention mask **is** the architecture. BERT and GPT share 95% of their PyTorch code; the only structural difference is whether the attention logits get the causal triangle applied or not, and that single bit determines whether the model is an encoder (understanding) or a decoder (generation).
- Pretraining objective and attention mask must agree: bidirectional encoder ↔ MLM; causal decoder ↔ next-token prediction. Mixing them breaks both the training signal and the inference story.
</details>

---

*End of Part C.*

---
## Summary

| Section | Key concept |
|---|---|
| **Part A §0** | Pretraining paradigm — self-supervised pretraining on massive unlabelled corpora, then fine-tuning on small labelled tasks |
| **Part A §1** | BERT = encoder-only + bidirectional attention + masked LM; `[CLS]` head for classification |
| **Part A §2** | GPT = decoder-only + causal (lower-triangular) attention mask + next-token prediction; same objective at train and inference |
| **Part A §3** | ViT = image → patches → linear projection → prepend `[CLS]` + learnable positional embedding → standard Transformer encoder |
| **Part A §4** | Discrete SSM: $h_{k+1} = \bar A h_k + \bar B u_k$, $y_k = C h_k$; unroll → $y = K*u$ with $K_i = C\bar A^i\bar B$ |
| **Part A §5** | Mamba = selective SSM: $\bar A, \bar B, C$ depend on $u_k$; kernel disappears; recovered by parallel scan |
| **Part B Task 1** | `ViTPatchEmbedding`: `nn.Unfold` → `nn.Linear` → prepend `[CLS]` token → add `pos_embed` |
| **Part B Task 2** | `ssm_recurrent` (RNN-style, $O(N)$ state) and `ssm_conv` (build $K$ incrementally, apply as 1-D causal convolution) — numerically identical |

**Take-aways:**
- Pretraining + fine-tuning is the default paradigm for modern Transformers; BERT, GPT, and ViT all share the encoder or decoder block but differ in mask + objective.
- ViT turns an image into a token sequence via patchification; positional encoding is *required* because self-attention is permutation-invariant.
- Linear SSMs are dually a recurrence (cheap inference) *and* a convolution (parallel training) — same weights, two views. The equivalence rests on $\bar A, \bar B, C$ being input-independent.
- Mamba trades the convolution form for input-dependent selectivity and recovers parallel training via a parallel scan. Exam Q2 and Q3 push on these two ideas.
